# Assignment 4

Ensamble models: specifically looking at the lightgbm model. 
Here, I will not need to do as much preprocessing as in Assignment 2, will have similar set-up/ prerpocessing as the trees model. 


In [2]:
import pandas as pd
import time
import numpy as np
import matplotlib.pyplot as plt
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import (f1_score, balanced_accuracy_score,
                             confusion_matrix, classification_report)

## Preprocessing

Things done:
- getting the 26 invalid rows removed from `is_ftp_login`.
- `service` missing values labelled as other
- dropping the `id` feature
- casting the nominal columns to category so lightgbm detects them automatically.


In [3]:
data = pd.read_csv("networkTraffic.csv", na_values= "?")
data = data.loc[data["is_ftp_login"].isin([0,1])].copy()
data["service"] = data["service"].fillna("other")
data = data.drop(columns= ["id"])

# Getting the nominal categories 
nominal = ["proto", "state", "service"]
for c in nominal:
    data[c] = data[c].astype("category")

# Getting the target variable and the predictors separate
X = data.drop(columns=["attack_cat"])
y = data["attack_cat"].to_numpy()
classes = np.unique(y)
num_classes = len(classes)

## Tuning Grid

In [5]:
seed = 42
estimator = LGBMClassifier(
    objective="multiclass",
    num_class=num_classes,
    random_state= seed ,
    n_jobs= 1 ,
    verbose=-1,
)

param_grid = {
    "class_weight":      [None, "balanced"],
    "learning_rate":     [0.05, 0.1],
    "n_estimators":      [300, 600],
    "num_leaves":        [31, 63],
    "min_child_samples": [20, 100],
}

outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=seed)

n_cand = int(np.prod([len(v) for v in param_grid.values()]))
print(f"{n_cand} candidates  ->  total fits = 5*(3*{n_cand}+1) = {5*(3*n_cand+1)}")


32 candidates  ->  total fits = 5*(3*32+1) = 485


In [6]:
# Nested cross validation set up
records, conf_matrices, cv_tables = [], [], []

for fold, (tr, te) in enumerate(outer_cv.split(X, y), start=1):
    X_tr, X_te = X.iloc[tr], X.iloc[te]
    y_tr, y_te = y[tr], y[te]

    search = GridSearchCV(estimator, param_grid, scoring="f1_macro",
                          cv=inner_cv, n_jobs=-1, refit=True, verbose=1)
    search.fit(X_tr, y_tr)
    y_pred = search.best_estimator_.predict(X_te)

    bp = search.best_params_
    records.append({
        "fold": fold,
        "macro_f1":     f1_score(y_te, y_pred, average="macro"),
        "weighted_f1":  f1_score(y_te, y_pred, average="weighted"),
        "balanced_acc": balanced_accuracy_score(y_te, y_pred),
        **{f"best_{k}": v for k, v in bp.items()},
    })
    conf_matrices.append(confusion_matrix(y_te, y_pred, labels=classes))

    ct = pd.DataFrame(search.cv_results_)
    ct["outer_fold"] = fold
    cv_tables.append(ct)

    print(f"\n--- outer fold {fold} ---", flush=True)
    print(records[-1], flush=True)
    print(classification_report(y_te, y_pred, zero_division=0), flush=True)

    pd.DataFrame(records).to_csv("lgbm_nested_cv_partial.csv", index=False)

results = pd.DataFrame(records)
results.to_csv("lgbm_nested_cv_results.csv", index=False)
pd.concat(cv_tables, ignore_index=True).to_csv("lgbm_cv_results_allfolds.csv", index=False)

agg_cm = pd.DataFrame(np.sum(conf_matrices, axis=0), index=classes, columns=classes)
agg_cm.to_csv("lgbm_confusion_matrix_aggregate.csv")
print("\nsaved: lgbm_nested_cv_results.csv, lgbm_cv_results_allfolds.csv, "
      "lgbm_confusion_matrix_aggregate.csv")



Fitting 3 folds for each of 32 candidates, totalling 96 fits

--- outer fold 1 ---
{'fold': 1, 'macro_f1': 0.5937483732620406, 'weighted_f1': 0.8049550520910961, 'balanced_acc': 0.6280969884360622, 'best_class_weight': 'balanced', 'best_learning_rate': 0.05, 'best_min_child_samples': 100, 'best_n_estimators': 300, 'best_num_leaves': 63}
              precision    recall  f1-score   support

           0       0.97      0.84      0.90     18600
           1       0.87      0.81      0.84      2797
           2       0.03      0.24      0.05       466
           3       0.36      0.34      0.35      3271
           4       0.88      0.55      0.68      8901
           5       0.04      0.14      0.07       535
           6       0.58      0.79      0.67      4849
           7       0.71      0.74      0.72        34
           8       0.55      0.85      0.66       303
           9       1.00      0.98      0.99     11774

    accuracy                           0.77     51530
   macro av

In [7]:
print("=== nested CV summary ===")
for m in ["macro_f1", "weighted_f1", "balanced_acc"]:
    print(f"{m:>14}: {results[m].mean():.4f} +/- {results[m].std():.4f}")

print("\n=== selected hyperparameters per fold ===")
print(results[[c for c in results.columns if c.startswith('best_')]].to_string(index=False))

print("\n=== aggregate confusion matrix (rows=true, cols=pred) ===")
print(agg_cm.to_string())

=== nested CV summary ===
      macro_f1: 0.5910 +/- 0.0096
   weighted_f1: 0.8041 +/- 0.0061
  balanced_acc: 0.6356 +/- 0.0164

=== selected hyperparameters per fold ===
best_class_weight  best_learning_rate  best_min_child_samples  best_n_estimators  best_num_leaves
         balanced                0.05                     100                300               63
         balanced                0.05                      20                600               31
         balanced                0.05                      20                300               31
         balanced                0.05                     100                300               63
         balanced                0.05                      20                300               31

=== aggregate confusion matrix (rows=true, cols=pred) ===
       0      1     2     3      4     5      6    7     8      9
0  77679     66     3   123    636   577  13622    1   281     10
1     16  11355   986   860    365   318     29   

In [8]:
allcv = pd.concat(cv_tables, ignore_index=True)

def avg_by(*params):
    cols = [f"param_{p}" for p in params]
    # dropna=False so the class_weight=None (unweighted) group is retained
    return (allcv.groupby(cols, dropna=False)["mean_test_score"].mean()
                 .reset_index().rename(columns={"mean_test_score": "mean_macro_f1"}))

print("=== Q1: class_weight ===")
q1 = avg_by("class_weight"); print(q1.to_string(index=False)); q1.to_csv("insight_q1_class_weight.csv", index=False)

print("\n=== Q2: learning_rate x n_estimators ===")
q2 = avg_by("learning_rate", "n_estimators"); print(q2.to_string(index=False)); q2.to_csv("insight_q2_lr_nest.csv", index=False)

print("\n=== Q3: num_leaves x min_child_samples ===")
q3 = avg_by("num_leaves", "min_child_samples"); print(q3.to_string(index=False)); q3.to_csv("insight_q3_regularisation.csv", index=False)


=== Q1: class_weight ===
param_class_weight  mean_macro_f1
          balanced       0.581500
               NaN       0.442798

=== Q2: learning_rate x n_estimators ===
 param_learning_rate  param_n_estimators  mean_macro_f1
                0.05                 300       0.563575
                0.05                 600       0.543133
                0.10                 300       0.501797
                0.10                 600       0.440091

=== Q3: num_leaves x min_child_samples ===
 param_num_leaves  param_min_child_samples  mean_macro_f1
               31                       20       0.517700
               31                      100       0.523950
               63                       20       0.486250
               63                      100       0.520696


In [10]:
best_cfg = (allcv.groupby([c for c in allcv.columns if c.startswith('param_')], dropna=False)
                 ["mean_test_score"].mean().idxmax())
best_params = {c.replace("param_", ""): v
               for c, v in zip([c for c in allcv.columns if c.startswith('param_')], best_cfg)}
print("final config (best mean inner macro-F1):", best_params)

final = LGBMClassifier(objective="multiclass", num_class=num_classes,
                       random_state=42, n_jobs= 1 , verbose=-1, **best_params)
final.fit(X, y)

imp = (pd.Series(final.feature_importances_, index=X.columns)
         .sort_values(ascending=False))
imp.to_csv("lgbm_feature_importance.csv")
print("\n=== top 15 feature importances ===")

final config (best mean inner macro-F1): {'class_weight': 'balanced', 'learning_rate': np.float64(0.05), 'min_child_samples': np.int64(100), 'n_estimators': np.int64(300), 'num_leaves': np.int64(63)}

=== top 15 feature importances ===


In [11]:
print(imp.head(15).to_string())

smean         17063
sbytes        16562
stcpb         10828
dtcpb         10274
ackdat         8704
synack         8494
sload          8430
ct_srv_src     6988
tcprtt         6979
dur            6402
ct_src_ltm     6210
djit           6089
dload          5944
sjit           5681
dmean          5653
